# Process simulation campaign with blueetl

In [1]:
from blueetl.analysis import run_from_file
analysis_config_file = "analysis_config_01_relative_with_soma.yaml"
loglevel = "INFO"
ma = run_from_file(analysis_config_file, loglevel=loglevel)
# ma = ma.apply_filter()

2025-05-06 11:04:04,664 INFO blueetl.analysis: MultiAnalyzer configuration: analysis_config_01_relative_with_soma.yaml
2025-05-06 11:04:04,695 INFO blueetl.cache: Loading cached checksums from analysis_output/spikes/config/checksums.cached.yaml
2025-05-06 11:04:04,696 INFO blueetl.cache: Initialize cache
2025-05-06 11:04:04,702 INFO blueetl.cache: Loading cached checksums from analysis_output/soma/config/checksums.cached.yaml
2025-05-06 11:04:04,703 INFO blueetl.cache: Initialize cache
2025-05-06 11:04:04,744 INFO blueetl.extract.simulations: Simulations extracted: 1/1, ids: [0]
2025-05-06 11:04:04,752 INFO blueetl.extract.simulations: Simulations extracted: 1/1, ids: [0]
2025-05-06 11:04:04,758 INFO blueetl.features: Step 1: grouping features by attributes...
2025-05-06 11:04:04,758 INFO blueetl.features: Preprocessing features 1/1 [id=0]
2025-05-06 11:04:04,758 INFO blueetl.features: Step 1: grouping features by attributes [DONE in 0.00 seconds]
2025-05-06 11:04:04,758 INFO blueetl.f

# Blueetl has created two analyzer objects for the spikes and soma blocks in the configuration file

In [2]:
# print(ma.analyzers)


# View the dataframes created for the spikes analyzer
- These are divided into "repo" and "features" dataframes.

- Repo dataframes are standard dataframes which organize the basic simualtion information and data.

- Features dataframes are calculated from the initial repo dataframes by code provided by the user, and typically store calcualted metrics or vectors such as PSTHs

In [3]:
spikes_analyzer = ma.analyzers["spikes"]

print(spikes_analyzer.repo.report.df)
print(spikes_analyzer.repo.neuron_classes.df)
print(spikes_analyzer.repo.simulations.df)
print(spikes_analyzer.repo.neurons.df)
print(spikes_analyzer.repo.windows.df)

print(spikes_analyzer.features.names)
print(spikes_analyzer.features.by_gid.df)
print(spikes_analyzer.features.by_gid_and_trial.df)
print(spikes_analyzer.features.by_neuron_class.df)
print(spikes_analyzer.features.by_neuron_class_and_trial.df)
print(spikes_analyzer.features.histograms.df)

   time  gid window  trial  simulation_id  circuit_id neuron_class
0   0.1    2     w1      0              0           0          All
1   0.2    0     w1      0              0           0          All
2   0.3    1     w1      0              0           0          All
3   0.7    2     w1      0              0           0          All
4   0.2    0     w1      0              0           0         L2_X
5   0.1    2     w1      0              0           0         L6_Y
6   0.3    1     w1      0              0           0         L6_Y
7   0.7    2     w1      0              0           0         L6_Y
   circuit_id neuron_class  count  limit population node_set  gids  \
0           0          All      3   1000    default     None  None   
1           0         L2_X      1   1000    default     None  None   
2           0         L6_Y      2   1000    default     None  None   

                 query  
0                   {}  
1  {"mtype": ["L2_X"]}  
2  {"mtype": ["L6_Y"]}  
     seed       

# Example blueetl filtering of dataframes

In [4]:
spikes_analyzer.repo.report.df.etl.q(neuron_class='L2_X', window='w2')


,time,gid,window,trial,simulation_id,circuit_id,neuron_class


# View the dataframes created for the soma analyzer

In [5]:
soma_report_analyzer = ma.analyzers["soma"]

print(soma_report_analyzer.repo.report.df)
print(soma_report_analyzer.repo.neuron_classes.df)
print(soma_report_analyzer.repo.simulations.df)
print(soma_report_analyzer.repo.neurons.df)
print(soma_report_analyzer.repo.windows.df)

print(soma_report_analyzer.features.names)
print(soma_report_analyzer.features.by_neuron_class.df)


   gid  time  value window  trial  simulation_id  circuit_id neuron_class
0    0   0.0    0.0     w1      0              0           0         L2_X
1    0   0.1    0.1     w1      0              0           0         L2_X
2    0   0.2    0.2     w1      0              0           0         L2_X
3    0   0.3    0.3     w1      0              0           0         L2_X
4    0   0.4    0.4     w1      0              0           0         L2_X
5    0   0.5    0.5     w1      0              0           0         L2_X
6    0   0.6    0.6     w1      0              0           0         L2_X
7    0   0.7    0.7     w1      0              0           0         L2_X
8    0   0.8    0.8     w1      0              0           0         L2_X
9    0   0.9    0.9     w1      0              0           0         L2_X
   circuit_id neuron_class  count  limit population node_set  gids  \
0           0         L2_X      1   1000    default     None  None   

                 query  
0  {"mtype": ["L2_X"

In [6]:
for feature_name in soma_report_analyzer.features.names:
    print(feature_name)
    print(soma_report_analyzer.features._data[feature_name].df)

by_neuron_class
                         neuron_class window  mean       std
simulation_id circuit_id                                    
0             0                  L2_X     w1  0.45  0.302765


Let's get the soma report dataframe to see the time and voltage (value) arrays. We will them to generate a timeseries trace array to extract eFEL features. Here, we create a dataframe `soma_df`to easily access the entries.

In [7]:
soma_report_analyzer.repo.report.df
soma_df = soma_report_analyzer.repo.report.df
soma_df

,gid,time,value,window,trial,simulation_id,circuit_id,neuron_class
0,0,0.0,0.0,w1,0,0,0,L2_X
1,0,0.1,0.1,w1,0,0,0,L2_X
2,0,0.2,0.2,w1,0,0,0,L2_X
3,0,0.3,0.3,w1,0,0,0,L2_X
4,0,0.4,0.4,w1,0,0,0,L2_X
5,0,0.5,0.5,w1,0,0,0,L2_X
6,0,0.6,0.6,w1,0,0,0,L2_X
7,0,0.7,0.7,w1,0,0,0,L2_X
8,0,0.8,0.8,w1,0,0,0,L2_X
9,0,0.9,0.9,w1,0,0,0,L2_X


Let's create a dictionary with the time and voltage arrays, start and end times.

In [8]:
import efel

# Extract time and voltage arrays
time = soma_df['time'].to_numpy()
voltage = soma_df['value'].to_numpy()

# Convert time to seconds
time = time * 1000  # Assuming time is in ms
# convert the voltage to mV
voltage = voltage * 1000

# Create trace dictionary for eFEL
trace = {
    'T': time,
    'V': voltage,
    'stim_start': [700],  # Adjust these values based on your simulation
    'stim_end': [2700]    # Adjust these values based on your simulation
}

# Create traces list and set eFEL settings
traces = [trace]
efel.set_setting('Threshold', -20)

# Extract features
feature_names = ['voltage_base']  # Add more features as needed
traces_results = efel.get_feature_values(traces, feature_names)

# Print results
for trace_results in traces_results:
    for feature_name, feature_values in trace_results.items():
        print(f"Feature {feature_name} has values: {feature_values}")

Feature voltage_base has values: [665.]


Similarly you can use other efel featuers on soma reports

In [9]:
efel.get_feature_names()

['ADP_peak_amplitude',
 'ADP_peak_indices',
 'ADP_peak_values',
 'AHP1_depth_from_peak',
 'AHP2_depth_from_peak',
 'AHP_depth',
 'AHP_depth_abs',
 'AHP_depth_abs_slow',
 'AHP_depth_diff',
 'AHP_depth_from_peak',
 'AHP_depth_slow',
 'AHP_slow_time',
 'AHP_time_from_peak',
 'AP1_amp',
 'AP1_begin_voltage',
 'AP1_begin_width',
 'AP1_peak',
 'AP1_width',
 'AP2_AP1_begin_width_diff',
 'AP2_AP1_diff',
 'AP2_AP1_peak_diff',
 'AP2_amp',
 'AP2_begin_voltage',
 'AP2_begin_width',
 'AP2_peak',
 'AP2_width',
 'AP_amplitude',
 'AP_amplitude_change',
 'AP_amplitude_diff',
 'AP_amplitude_from_voltagebase',
 'AP_begin_indices',
 'AP_begin_time',
 'AP_begin_voltage',
 'AP_begin_width',
 'AP_duration',
 'AP_duration_change',
 'AP_duration_half_width',
 'AP_duration_half_width_change',
 'AP_end_indices',
 'AP_fall_indices',
 'AP_fall_rate',
 'AP_fall_rate_change',
 'AP_fall_time',
 'AP_height',
 'AP_peak_downstroke',
 'AP_peak_upstroke',
 'AP_phaseslope',
 'AP_rise_indices',
 'AP_rise_rate',
 'AP_rise_ra